# M4 — Üretilmiş afet görselleri · üretim defteri

**KrizKalkan AI · Colab A100**

Bu defter, M4'ün **kriz alanındaki kör noktasını** kapatmak için gereken
pozitif sınıf verisini üretir.

---

## Neden bu defter var

M4 ölçüldü ve alan içinde çalışmadığı bulundu:

| Soru | Küme | Sonuç |
|---|---|---|
| Gerçek afet fotoğrafına "sentetik" der mi? | Wikimedia afet korpusu (n=786) | 9/786 — çok iyi |
| **Üretilmiş afet fotoğrafını yakalar mı?** | Üretilmiş korpus (n=35) | **0/35 — hiç** |

Model bozuk değil: OpenFake'te AUC 0,8564, sahtelerde medyan skor 0,5152.
Kör nokta **alana özgü**.

Sebep eğitim kümesinin kuruluşunda: afet korpusu **negatif** sınıfa kondu,
pozitif sınıfta hiç afet içeriği yoktu. Model büyük olasılıkla
"afet sahnesi → gerçek" kısayolunu öğrendi. Yanlış alarmların 782/786'dan
9/786'ya düşmesiyle bu kör nokta aynı madalyonun iki yüzü.

Düzeltme: **pozitif sınıfa üretilmiş afet görseli koymak.** Bu defter onu üretir.

---

## Tasarımın kritik noktası: üretici bazlı ayrım

Üretilmiş görselle eğitip üretilmiş görselle ölçmek **alan içi başarım**
ölçmek olurdu ve bu depo onu kabul etmiyor (bkz. `docs/metrikler/m4.md`).
Bu yüzden üreticiler role ayrılır ve roller KARIŞMAZ:

| Üretici | Mimari | Lisans | Rol |
|---|---|---|---|
| SANA 1.6B | linear-attention DiT | **Apache 2.0** | eğitim |
| SDXL base 1.0 | UNet latent diffusion | OpenRAIL++-M | eğitim |
| **Hizalı sahteler** (kendi afet fotoğraflarımızdan) | SDXL VAE + img2img | kendi korpusumuz | eğitim |
| PixArt-Σ XL-2 | DiT + T5 | OpenRAIL++-M | **tutulan** (ölçüm) |
| z_image (Tongyi-MAI) | — (API) | — | **tutulan** (elde 35 görsel var) |

> **FLUX.1-schnell neden yok?** Apache 2.0 ve mimari olarak ideal, ama HF'de
> kapılı (`gated`): lisansı kabul edip Colab'a `HF_TOKEN` eklemek gerekiyor.
> SANA da Apache 2.0, DiT ailesinden ve **kapısız**. FLUX'ı eklemek isterseniz
> depoda kabul edip token tanımlayın, sonra SANA hücresini kopyalayıp
> `FluxPipeline` ile değiştirin.
>
> **SD 2.1 neden yok?** Stability, `stabilityai/stable-diffusion-2-1-base`
> deposunu HF'den kaldırmış (`RepositoryNotFoundError`). Yerine PixArt-Σ
> kondu — üstelik DiT olduğu için mimari çeşitliliği artırıyor.

Eğitim sonrası ölçüm yalnızca *tutulan* üreticilerde yapılır: o zaman ölçülen
şey **görülmemiş üreticiye aktarım** olur, ezber değil.

---

## Hizalı sahteler: kör noktayı asıl kapatan parça

Kör noktanın sebebi **içerik yanlılığı**: model görselin konusuna bakıp karar
veriyor. Afet sahnesi yalnızca "gerçek" sınıfında göründüğü için "afet → gerçek"
kısayolunu öğrendi.

Yeni üretilmiş afet görselleri bunu kısmen düzeltir ama tam çözmez: üretilmiş
görsellerin konusu da, kompozisyonu da gerçeklerden farklıdır, dolayısıyla
model yine içerikten yararlanabilir.

Tam çözüm, sahteyi **gerçek fotoğrafın kendisinden** üretmektir (Aligned
Datasets · ICLR 2025; B-Free · CVPR 2025). İki sınıfta içerik birebir aynı olur;
geriye kalan tek fark üretim izidir. Model başka hiçbir ipucu kullanamaz.

Yerelde ölçüldü (pilot, n=7): mevcut model, kendi VAE'sinden geçmiş afet
fotoğraflarını gerçeklerinden **AUC 0,5612** ile ayırıyor — yani neredeyse hiç.
Bu, teşhisi doğruluyor.

Bu defter iki hizalı sahte üretir:

| Yöntem | Ne yapar | Güç |
|---|---|---|
| VAE yeniden kurma | Fotoğraf SDXL sıkıştırıcısından geçip geri açılır | zayıf iz, çok ucuz |
| img2img (güç 0,35) | Fotoğraf kısmen yeniden üretilir | güçlü iz, içerik korunur |

> **Lisans notu.** Eğitimde kullanılan üreticinin lisansı ağırlığın köken
> zincirine girer. `genis` modeli tam da bu yüzden devreye alınmamıştı
> (OpenFakeTiny · CC BY-NC). FLUX.1-schnell **Apache 2.0**'dır ve zinciri
> temiz bırakır. SDXL'in OpenRAIL++-M lisansı kullanım kısıtları taşır
> (Ek A dezenformasyon üretimini yasaklar — biz dedektör eğitiyoruz, tersi).
> Zinciri tartışmasız tutmak isterseniz `SADECE_TEMIZ_LISANS = True` yapın:
> yalnızca Apache 2.0 üreticiler çalışır.

---

## Etik

Komutların hiçbiri gerçek kişi, gerçek olay adı veya kurum içermez. Görseller
yalnızca dedektör eğitimi ve ölçümünde kullanılır, gerçek olay diye
etiketlenmez, dosya adları `SYNTH_` ile başlar ve depoya girmez.
Kayıt: `docs/etik-protokol.md` §6.3.

---

## Çalıştırma

Hücreleri sırayla çalıştırın. Her üretici ayrı hücrededir: biri çökerse
diğerleri etkilenmez, hafızadan düşürülüp devam edilir. Sonunda tek bir zip
iner.

## 1 · Ayarlar

In [ ]:
# ── Üretim ayarları ────────────────────────────────────────────────────────
TOHUM = 20260915          # tekrarlanabilirlik: aynı tohum aynı görselleri verir
SADECE_TEMIZ_LISANS = False   # True → yalnızca Apache 2.0 üreticiler

# Üretici başına görsel sayısı. Eğitim için toplam ~1200 pozitif hedefliyoruz;
# tutulan küme ölçüm için daha küçük olabilir ama n<150 güven aralığını
# kullanışsız hâle getirir.
ADET = {
    "sana-1600m":   600,   # eğitim  · Apache 2.0
    "sdxl-base":    600,   # eğitim  · OpenRAIL++-M
    "pixart-sigma": 250,   # TUTULAN · ölçüm
}

# Hizalı sahteler kaç afet fotoğrafından üretilsin (0 = kapalı).
# Yalnızca EĞİTİM olaylarından üretilir; tutulan olaylara dokunulmaz.
HIZALI_ADET = 500

# Tür dağılımı gerçek korpusu taklit eder (Kahramanmaraş ağırlıklı).
TUR_AGIRLIK = {"deprem": 0.55, "sel": 0.22, "yangın": 0.23}

CIKTI_DIZINI = "/content/sentetik_afet"
# ───────────────────────────────────────────────────────────────────────────

import os, json, random, math
os.makedirs(CIKTI_DIZINI, exist_ok=True)
os.makedirs(f"{CIKTI_DIZINI}/goruntuler", exist_ok=True)
print("çıktı:", CIKTI_DIZINI)
print("toplam hedef:", sum(ADET.values()), "görsel")

## 2 · Kurulum

In [ ]:
!pip -q install "diffusers==0.36.0" "transformers==4.57.1" "accelerate==1.10.1" \
                "sentencepiece==0.2.1" "protobuf==6.33.1" 2>&1 | tail -2

import torch

assert torch.cuda.is_available(), (
    "GPU çalışma zamanı seçin: Runtime → Change runtime type → A100"
)

_ozellik = torch.cuda.get_device_properties(0)
_vram = _ozellik.total_memory / 1e9   # not: total_mem DEĞİL

print("torch  ", torch.__version__)
print("GPU    ", torch.cuda.get_device_name(0))
print("VRAM   ", f"{_vram:.0f} GB")

# FLUX.1-schnell bfloat16'da ~24 GB ister. 40 GB'da cpu offload ile rahat
# çalışır; 16 GB'lık bir kartta (T4/V100) sığmaz ve hücre OOM ile düşer.
if _vram < 30:
    print()
    print(f"⚠ {_vram:.0f} GB — FLUX hücresi sığmayabilir. Seçenekler:")
    print("   • A100 çalışma zamanına geçin (önerilen), ya da")
    print("   • FLUX hücresini atlayıp SDXL + SD 2.1 ile devam edin")

## 3 · Komut bankası ve biçim profili

İkisi de depodan **gömülerek** gelir; Colab'da depoyu klonlamaya gerek yok.

**Biçim profili neden gerekli.** Üretici modeller pırıl pırıl PNG verir; gerçek
korpus ise %95 JPEG, bayt/piksel medyanı 0,292, genişlik medyanı 960'tır. Bu
fark düzeltilmezse model "temiz dosya = üretilmiş" kısayolunu öğrenir. Ölçüldü:
düzeltme öncesi **sadece bayt/piksel ile ayrım AUC 0,8500**, sonrası **0,5005**.

Bu yüzden her görsel, gerçek korpustan çekilen bir (genişlik, bayt/piksel)
hedefine oturtulur ve **JPEG olarak tek kez** kodlanır. PNG indirip yerelde
kodlamak her dosyaya ikinci bir kodlama bindirirdi; çift sıkıştırma izi başlı
başına bir karıştırıcıdır.

In [ ]:
KOMUTLAR = {
 "_aciklama": "M4/M6 için ÜRETİLMİŞ afet görselleri korpusunun komut bankası. Her komut fotoğrafçılık diliyle yazılmıştır ('amateur smartphone photo', 'press photograph'): tehdit modeli, sosyal medyada haber fotoğrafı diye dolaşan üretilmiş görseldir; konsept sanatı değildir. 'cinematic', '8k', 'artstation' gibi ifadeler BİLİNÇLİ OLARAK yoktur — modeli sanatsal üretime iterler ve ölçülmek istenen alanı kaçırırlar.",
 "_etik": "Hiçbir komut gerçek kişi, gerçek olay adı veya kurum logosu içermez. Görseller yalnızca dedektör değerlendirmesinde kullanılır, gerçek olay diye etiketlenmez ve depoya gömülmez (data/ .gitignore altındadır). Kayıt: docs/etik-protokol.md",
 "_kaynak": "Komutlar takım tarafından yazılmıştır; hiçbir veri kümesinden kopyalanmamıştır (lisans zinciri temiz kalsın diye).",
 "deprem": [
  "candid documentary photograph, unstaged, taken by a passer-by, collapsed six-storey concrete apartment building, pancaked floors, twisted rebar, thick concrete dust in the air, overcast morning light, handheld and slightly tilted, Mediterranean town street",
  "press photograph, search and rescue team in orange high-visibility vests climbing over a rubble pile of a collapsed building, dust, scattered household belongings, grey daylight",
  "candid documentary photograph at dusk, street of damaged apartment blocks after an earthquake, facades sheared off exposing rooms, cars crushed under debris, people standing in the street wrapped in blankets",
  "handheld news photo, wide shot of an earthquake-damaged neighbourhood, several partially collapsed mid-rise buildings, cranes and excavators working on debris, hazy sky",
  "candid documentary photograph, interior of an apartment after a strong earthquake, cracked plaster walls, fallen bookshelves and broken crockery across the floor, daylight through a window",
  "documentary photograph, night scene at a collapsed building, portable floodlights on tripods, rescue workers with helmets and headlamps searching rubble, dust in the light beams",
  "candid snapshot, tent encampment for displaced residents in a stadium car park after an earthquake, rows of white and blue tents, laundry lines, winter light",
  "press photo, close view of a cracked and buckled asphalt road with a wide fissure after a seismic event, kerbstones displaced, parked cars tilted",
  "candid documentary photograph, unstaged, minaret of a small town mosque snapped and fallen across a street, rubble at the base, onlookers at a distance, flat daylight",
  "handheld photo, collapsed roof of a village house built of stone and timber, debris spilling into a courtyard, chickens and scattered furniture, morning light",
  "news photograph, excavator clearing rubble of a destroyed building while rescue workers watch, dust cloud, high-visibility clothing, overcast",
  "candid photograph from a balcony, view over a street of earthquake damage, cracked facades, ambulance with lights on, people gathered on the pavement",
  "documentary photo, row of damaged shops with collapsed awnings and shattered glass storefronts, merchandise in the street, grey afternoon",
  "candid documentary photograph, drone-style elevated view of a town block with three collapsed buildings among intact ones, debris fields, emergency vehicles on the access road",
  "press photograph, close-up of rescue dog and handler working on a rubble pile, dust, high-visibility vest, low winter sun"
 ],
 "sel": [
  "candid documentary photograph, unstaged, flooded town street with brown muddy water up to car windows, partially submerged vehicles, shopfronts flooded, overcast light",
  "press photograph, rescue boat with inflatable hull carrying residents through a flooded residential street, water reaching first-floor windows, grey sky",
  "handheld documentary photograph, torrential flood water rushing through a narrow street between stone buildings, debris and furniture carried by the current, daytime",
  "documentary photo, aftermath of a flash flood in a mountain village, mud and gravel covering the road, damaged cars pushed against walls, clearing sky",
  "candid documentary photograph, interior of a flooded ground-floor shop, muddy water line on the walls, ruined stock floating, fluorescent light",
  "news photograph, collapsed bridge over a swollen brown river after heavy rain, broken concrete span, fast water, riverside trees bent",
  "candid snapshot, flooded agricultural field with submerged greenhouses and a tractor stuck in water, flat horizon, overcast",
  "press photo, residents wading through knee-deep flood water carrying belongings above their heads, narrow urban street, rain",
  "handheld photo, flood water receding from a street leaving thick mud, damaged parked cars, shopkeepers shovelling, harsh midday sun",
  "documentary photograph, storm drain overwhelmed and water surging up through a road, pedestrians on a raised kerb, heavy rain, grey light",
  "candid documentary photograph, unstaged, seaside promenade during a storm surge, waves breaking over the barrier onto the road, spray, dark clouds"
 ],
 "yangın": [
  "candid documentary photograph, unstaged, forest fire on a hillside at dusk, orange flames along the ridge, heavy smoke, silhouetted pine trees, handheld",
  "press photograph, firefighters in protective gear with a hose line at the edge of a burning pine forest, smoke, embers, low visibility",
  "handheld documentary photograph, wildfire approaching a village, orange glow behind houses, thick smoke obscuring the sky, residents watching from a road",
  "documentary photo, aerial-style view of a firefighting aircraft dropping water over a burning forest, smoke plume, hazy air",
  "candid documentary photograph, burnt hillside after a wildfire, blackened tree trunks, ash-covered ground, smoke still rising in places, harsh daylight",
  "news photograph, apartment block fire at night, flames from several windows, fire engine with ladder extended, water spray, dark sky",
  "candid snapshot, thick brown smoke plume from a wildfire seen from a coastal road, sun reddened by smoke, parked cars, hot afternoon",
  "press photo, firefighters damping down smouldering remains of a burnt house, charred beams, ash, protective helmets, daylight",
  "handheld photo, wildfire crossing a rural road at night, flames on both verges, orange light on the asphalt, smoke",
  "documentary photograph, burnt-out cars and blackened olive trees after a fire passed through a rural property, ash, grey overcast",
  "candid documentary photograph, unstaged, market stall fire in a town street, flames and black smoke, bystanders with phones, daylight"
 ],
 "_duzeltme": "Pilot koşuda 'amateur smartphone photo' ifadesi 3 görselin 2'sinde kadraja telefon tutan bir el koydurdu — istenen çerçeveleme değil. İfadeler fotojurnalizm diline çevrildi; cihazı göstermeden aynı amatör çerçevelemeyi veriyor."
}

BICIM_PROFILI = {
 "_aciklama": "Gerçek Türk afet korpusunun (genişlik, bayt/piksel) çiftleri. Üretilmiş görseller bu dağılımdan çekilen hedeflere oturtulur; aksi hâlde biçim farkı tek başına sınıfı ele veriyor (ölçüldü: AUC 0,8500 → oturtma sonrası 0,5005).",
 "_kaynak": "data/external/provenance/goruntuler · JPEG dosyalar",
 "_olcum": {
  "n": 747,
  "genislik_medyan": 960,
  "bayt_piksel_medyan": 0.2924
 },
 "ciftler": [
  [
   960,
   0.3946
  ],
  [
   960,
   0.2416
  ],
  [
   960,
   0.2842
  ],
  [
   960,
   0.3694
  ],
  [
   960,
   0.4851
  ],
  [
   960,
   0.3513
  ],
  [
   960,
   0.422
  ],
  [
   960,
   0.378
  ],
  [
   960,
   0.4779
  ],
  [
   960,
   0.3941
  ],
  [
   960,
   0.3192
  ],
  [
   960,
   0.3512
  ],
  [
   960,
   0.3796
  ],
  [
   960,
   0.4351
  ],
  [
   960,
   0.3917
  ],
  [
   960,
   0.4657
  ],
  [
   960,
   0.2717
  ],
  [
   960,
   0.3635
  ],
  [
   960,
   0.2799
  ],
  [
   960,
   0.2747
  ],
  [
   960,
   0.4755
  ],
  [
   960,
   0.4727
  ],
  [
   960,
   0.2603
  ],
  [
   960,
   0.4498
  ],
  [
   960,
   0.4568
  ],
  [
   960,
   0.3717
  ],
  [
   960,
   0.2516
  ],
  [
   960,
   0.3133
  ],
  [
   960,
   0.3656
  ],
  [
   960,
   0.329
  ],
  [
   960,
   0.3563
  ],
  [
   960,
   0.3699
  ],
  [
   960,
   0.423
  ],
  [
   960,
   0.2454
  ],
  [
   960,
   0.1772
  ],
  [
   960,
   0.556
  ],
  [
   960,
   0.5459
  ],
  [
   960,
   0.4341
  ],
  [
   960,
   0.1814
  ],
  [
   960,
   0.3557
  ],
  [
   960,
   0.3273
  ],
  [
   960,
   0.3844
  ],
  [
   960,
   0.2372
  ],
  [
   960,
   0.3457
  ],
  [
   960,
   0.4092
  ],
  [
   960,
   0.3881
  ],
  [
   960,
   0.4521
  ],
  [
   960,
   0.4751
  ],
  [
   960,
   0.3543
  ],
  [
   960,
   0.4045
  ],
  [
   960,
   0.4727
  ],
  [
   960,
   0.2971
  ],
  [
   960,
   0.1812
  ],
  [
   768,
   0.093
  ],
  [
   960,
   0.2105
  ],
  [
   960,
   0.1439
  ],
  [
   960,
   0.1496
  ],
  [
   960,
   0.2748
  ],
  [
   960,
   0.3269
  ],
  [
   960,
   0.3676
  ],
  [
   960,
   0.2689
  ],
  [
   960,
   0.3596
  ],
  [
   960,
   0.469
  ],
  [
   960,
   0.4502
  ],
  [
   960,
   0.3114
  ],
  [
   960,
   0.3128
  ],
  [
   960,
   0.3725
  ],
  [
   960,
   0.1666
  ],
  [
   960,
   0.3172
  ],
  [
   960,
   0.2289
  ],
  [
   960,
   0.2963
  ],
  [
   960,
   0.2212
  ],
  [
   960,
   0.3893
  ],
  [
   960,
   0.2394
  ],
  [
   960,
   0.2327
  ],
  [
   960,
   0.2486
  ],
  [
   960,
   0.2893
  ],
  [
   960,
   0.2966
  ],
  [
   960,
   0.3954
  ],
  [
   960,
   0.275
  ],
  [
   960,
   0.2169
  ],
  [
   960,
   0.3628
  ],
  [
   960,
   0.3056
  ],
  [
   960,
   0.2738
  ],
  [
   960,
   0.3515
  ],
  [
   960,
   0.4244
  ],
  [
   960,
   0.4022
  ],
  [
   960,
   0.3868
  ],
  [
   960,
   0.3157
  ],
  [
   960,
   0.5427
  ],
  [
   960,
   0.2575
  ],
  [
   960,
   0.5034
  ],
  [
   721,
   0.3606
  ],
  [
   960,
   0.3355
  ],
  [
   960,
   0.2951
  ],
  [
   960,
   0.3935
  ],
  [
   960,
   0.6085
  ],
  [
   720,
   0.7174
  ],
  [
   720,
   0.7039
  ],
  [
   960,
   0.1735
  ],
  [
   720,
   0.3017
  ],
  [
   960,
   0.5253
  ],
  [
   960,
   0.4575
  ],
  [
   960,
   0.3964
  ],
  [
   960,
   0.5696
  ],
  [
   960,
   0.3073
  ],
  [
   960,
   0.3912
  ],
  [
   960,
   0.496
  ],
  [
   960,
   0.1568
  ],
  [
   960,
   0.4562
  ],
  [
   960,
   0.3696
  ],
  [
   960,
   0.4519
  ],
  [
   960,
   0.3975
  ],
  [
   960,
   0.5264
  ],
  [
   960,
   0.3259
  ],
  [
   960,
   0.3447
  ],
  [
   960,
   0.3935
  ],
  [
   960,
   0.3758
  ],
  [
   960,
   0.3024
  ],
  [
   960,
   0.3718
  ],
  [
   800,
   0.3748
  ],
  [
   800,
   0.444
  ],
  [
   960,
   0.3481
  ],
  [
   960,
   0.3282
  ],
  [
   960,
   0.3964
  ],
  [
   960,
   0.4479
  ],
  [
   960,
   0.3945
  ],
  [
   960,
   0.2699
  ],
  [
   960,
   0.3043
  ],
  [
   960,
   0.3882
  ],
  [
   960,
   0.339
  ],
  [
   960,
   0.3387
  ],
  [
   960,
   0.4006
  ],
  [
   960,
   0.408
  ],
  [
   960,
   0.3931
  ],
  [
   960,
   0.4505
  ],
  [
   960,
   0.4159
  ],
  [
   960,
   0.2986
  ],
  [
   960,
   0.4118
  ],
  [
   960,
   0.3429
  ],
  [
   960,
   0.3736
  ],
  [
   960,
   0.3377
  ],
  [
   960,
   0.3518
  ],
  [
   960,
   0.4397
  ],
  [
   960,
   0.3766
  ],
  [
   960,
   0.5014
  ],
  [
   960,
   0.4967
  ],
  [
   960,
   0.1437
  ],
  [
   960,
   0.321
  ],
  [
   960,
   0.2698
  ],
  [
   960,
   0.2863
  ],
  [
   960,
   0.1375
  ],
  [
   960,
   0.2129
  ],
  [
   960,
   0.2307
  ],
  [
   960,
   0.3293
  ],
  [
   960,
   0.3176
  ],
  [
   960,
   0.1764
  ],
  [
   960,
   0.3262
  ],
  [
   960,
   0.2703
  ],
  [
   960,
   0.2806
  ],
  [
   960,
   0.2088
  ],
  [
   960,
   0.4388
  ],
  [
   960,
   0.2505
  ],
  [
   960,
   0.3366
  ],
  [
   960,
   0.3908
  ],
  [
   960,
   0.4248
  ],
  [
   960,
   0.3193
  ],
  [
   960,
   0.2823
  ],
  [
   960,
   0.3881
  ],
  [
   960,
   0.2343
  ],
  [
   960,
   0.3323
  ],
  [
   960,
   0.2523
  ],
  [
   960,
   0.3857
  ],
  [
   960,
   0.4776
  ],
  [
   960,
   0.3649
  ],
  [
   960,
   0.2952
  ],
  [
   960,
   0.3397
  ],
  [
   960,
   0.3617
  ],
  [
   960,
   0.3607
  ],
  [
   960,
   0.3521
  ],
  [
   960,
   0.3646
  ],
  [
   960,
   0.3002
  ],
  [
   960,
   0.2865
  ],
  [
   960,
   0.1795
  ],
  [
   960,
   0.2665
  ],
  [
   960,
   0.407
  ],
  [
   960,
   0.4705
  ],
  [
   960,
   0.3214
  ],
  [
   960,
   0.4968
  ],
  [
   960,
   0.5031
  ],
  [
   960,
   0.5363
  ],
  [
   960,
   0.3266
  ],
  [
   960,
   0.5337
  ],
  [
   960,
   0.4614
  ],
  [
   960,
   0.4424
  ],
  [
   960,
   0.4237
  ],
  [
   960,
   0.3522
  ],
  [
   960,
   0.4187
  ],
  [
   960,
   0.4502
  ],
  [
   960,
   0.3326
  ],
  [
   960,
   0.2013
  ],
  [
   960,
   0.5061
  ],
  [
   960,
   0.4693
  ],
  [
   960,
   0.4686
  ],
  [
   960,
   0.4455
  ],
  [
   960,
   0.3022
  ],
  [
   960,
   0.2599
  ],
  [
   960,
   0.2053
  ],
  [
   960,
   0.169
  ],
  [
   960,
   0.1418
  ],
  [
   960,
   0.1786
  ],
  [
   960,
   0.2083
  ],
  [
   960,
   0.2549
  ],
  [
   960,
   0.2248
  ],
  [
   960,
   0.2875
  ],
  [
   960,
   0.2895
  ],
  [
   960,
   0.3526
  ],
  [
   960,
   0.2523
  ],
  [
   500,
   0.3606
  ],
  [
   500,
   0.4101
  ],
  [
   500,
   0.4739
  ],
  [
   500,
   0.4985
  ],
  [
   500,
   0.1921
  ],
  [
   500,
   0.4045
  ],
  [
   960,
   0.4071
  ],
  [
   960,
   0.2893
  ],
  [
   960,
   0.2696
  ],
  [
   960,
   0.2098
  ],
  [
   960,
   0.2599
  ],
  [
   960,
   0.3754
  ],
  [
   960,
   0.2584
  ],
  [
   960,
   0.0711
  ],
  [
   960,
   0.2714
  ],
  [
   960,
   0.2884
  ],
  [
   960,
   0.3117
  ],
  [
   960,
   0.2483
  ],
  [
   960,
   0.2354
  ],
  [
   960,
   0.2916
  ],
  [
   960,
   0.3604
  ],
  [
   960,
   0.3211
  ],
  [
   960,
   0.2888
  ],
  [
   960,
   0.3334
  ],
  [
   960,
   0.2409
  ],
  [
   960,
   0.2869
  ],
  [
   960,
   0.2653
  ],
  [
   960,
   0.291
  ],
  [
   960,
   0.3237
  ],
  [
   960,
   0.2184
  ],
  [
   960,
   0.2382
  ],
  [
   960,
   0.3297
  ],
  [
   960,
   0.2365
  ],
  [
   960,
   0.5405
  ],
  [
   960,
   0.5559
  ],
  [
   960,
   0.2402
  ],
  [
   960,
   0.3163
  ],
  [
   960,
   0.2415
  ],
  [
   960,
   0.195
  ],
  [
   960,
   0.4108
  ],
  [
   960,
   0.224
  ],
  [
   960,
   0.2252
  ],
  [
   960,
   0.3064
  ],
  [
   960,
   0.3466
  ],
  [
   960,
   0.321
  ],
  [
   960,
   0.3096
  ],
  [
   960,
   0.2651
  ],
  [
   960,
   0.3202
  ],
  [
   960,
   0.3547
  ],
  [
   960,
   0.3391
  ],
  [
   960,
   0.0919
  ],
  [
   960,
   0.1223
  ],
  [
   960,
   0.2444
  ],
  [
   960,
   0.5236
  ],
  [
   960,
   0.2611
  ],
  [
   960,
   0.2374
  ],
  [
   960,
   0.3978
  ],
  [
   960,
   0.2018
  ],
  [
   960,
   0.4755
  ],
  [
   960,
   0.3265
  ],
  [
   960,
   0.2815
  ],
  [
   960,
   0.3028
  ],
  [
   960,
   0.349
  ],
  [
   960,
   0.3169
  ],
  [
   960,
   0.3439
  ],
  [
   960,
   0.329
  ],
  [
   960,
   0.3253
  ],
  [
   960,
   0.2417
  ],
  [
   960,
   0.3174
  ],
  [
   960,
   0.2492
  ],
  [
   960,
   0.3419
  ],
  [
   960,
   0.3153
  ],
  [
   960,
   0.3008
  ],
  [
   960,
   0.2179
  ],
  [
   960,
   0.2088
  ],
  [
   960,
   0.211
  ],
  [
   960,
   0.3436
  ],
  [
   960,
   0.2294
  ],
  [
   960,
   0.3084
  ],
  [
   960,
   0.2613
  ],
  [
   960,
   0.1539
  ],
  [
   960,
   0.1792
  ],
  [
   960,
   0.2304
  ],
  [
   960,
   0.297
  ],
  [
   960,
   0.2133
  ],
  [
   960,
   0.4091
  ],
  [
   960,
   0.1985
  ],
  [
   960,
   0.208
  ],
  [
   960,
   0.4414
  ],
  [
   960,
   0.1884
  ],
  [
   960,
   0.3604
  ],
  [
   960,
   0.1554
  ],
  [
   960,
   0.2667
  ],
  [
   960,
   0.2831
  ],
  [
   960,
   0.1158
  ],
  [
   960,
   0.2242
  ],
  [
   960,
   0.1692
  ],
  [
   960,
   0.2022
  ],
  [
   960,
   0.4922
  ],
  [
   960,
   0.42
  ],
  [
   960,
   0.4598
  ],
  [
   960,
   0.5114
  ],
  [
   960,
   0.5111
  ],
  [
   960,
   0.2216
  ],
  [
   960,
   0.3584
  ],
  [
   960,
   0.2577
  ],
  [
   960,
   0.1556
  ],
  [
   960,
   0.2622
  ],
  [
   960,
   0.311
  ],
  [
   960,
   0.2856
  ],
  [
   960,
   0.2054
  ],
  [
   960,
   0.392
  ],
  [
   960,
   0.1941
  ],
  [
   960,
   0.1945
  ],
  [
   960,
   0.1546
  ],
  [
   960,
   0.1252
  ],
  [
   960,
   0.1535
  ],
  [
   960,
   0.0951
  ],
  [
   960,
   0.4019
  ],
  [
   960,
   0.3889
  ],
  [
   960,
   0.3159
  ],
  [
   960,
   0.2942
  ],
  [
   960,
   0.3872
  ],
  [
   960,
   0.2444
  ],
  [
   960,
   0.2903
  ],
  [
   960,
   0.3001
  ],
  [
   960,
   0.1677
  ],
  [
   960,
   0.1151
  ],
  [
   960,
   0.3279
  ],
  [
   960,
   0.2283
  ],
  [
   960,
   0.25
  ],
  [
   960,
   0.1421
  ],
  [
   960,
   0.1934
  ],
  [
   960,
   0.1357
  ],
  [
   960,
   0.1489
  ],
  [
   960,
   0.2152
  ],
  [
   960,
   0.2846
  ],
  [
   960,
   0.1903
  ],
  [
   960,
   0.2153
  ],
  [
   960,
   0.153
  ],
  [
   960,
   0.3012
  ],
  [
   960,
   0.2459
  ],
  [
   960,
   0.3256
  ],
  [
   960,
   0.3244
  ],
  [
   960,
   0.3084
  ],
  [
   960,
   0.5453
  ],
  [
   960,
   0.3635
  ],
  [
   960,
   0.3526
  ],
  [
   960,
   0.3437
  ],
  [
   960,
   0.3414
  ],
  [
   960,
   0.333
  ],
  [
   960,
   0.3256
  ],
  [
   960,
   0.4029
  ],
  [
   960,
   0.452
  ],
  [
   960,
   0.4586
  ],
  [
   960,
   0.417
  ],
  [
   960,
   0.4362
  ],
  [
   960,
   0.4085
  ],
  [
   960,
   0.3357
  ],
  [
   960,
   0.387
  ],
  [
   960,
   0.3165
  ],
  [
   960,
   0.2396
  ],
  [
   960,
   0.3171
  ],
  [
   960,
   0.258
  ],
  [
   960,
   0.3607
  ],
  [
   960,
   0.333
  ],
  [
   960,
   0.2804
  ],
  [
   960,
   0.3022
  ],
  [
   960,
   0.353
  ],
  [
   960,
   0.4196
  ],
  [
   960,
   0.223
  ],
  [
   960,
   0.2281
  ],
  [
   960,
   0.3349
  ],
  [
   960,
   0.2366
  ],
  [
   960,
   0.2284
  ],
  [
   960,
   0.2131
  ],
  [
   960,
   0.3814
  ],
  [
   960,
   0.3576
  ],
  [
   960,
   0.3131
  ],
  [
   960,
   0.3334
  ],
  [
   960,
   0.3335
  ],
  [
   960,
   0.2862
  ],
  [
   960,
   0.311
  ],
  [
   960,
   0.3565
  ],
  [
   960,
   0.4693
  ],
  [
   960,
   0.4269
  ],
  [
   960,
   0.5155
  ],
  [
   960,
   0.4437
  ],
  [
   960,
   0.4276
  ],
  [
   960,
   0.3517
  ],
  [
   960,
   0.2331
  ],
  [
   960,
   0.2996
  ],
  [
   960,
   0.4227
  ],
  [
   960,
   0.3406
  ],
  [
   960,
   0.3173
  ],
  [
   960,
   0.4693
  ],
  [
   960,
   0.5014
  ],
  [
   960,
   0.4918
  ],
  [
   960,
   0.3457
  ],
  [
   960,
   0.2661
  ],
  [
   960,
   0.3353
  ],
  [
   960,
   0.3466
  ],
  [
   960,
   0.3897
  ],
  [
   960,
   0.3918
  ],
  [
   960,
   0.2273
  ],
  [
   960,
   0.5289
  ],
  [
   960,
   0.3869
  ],
  [
   960,
   0.3615
  ],
  [
   960,
   0.3555
  ],
  [
   960,
   0.3084
  ],
  [
   960,
   0.4197
  ],
  [
   960,
   0.2657
  ],
  [
   960,
   0.2473
  ],
  [
   960,
   0.3179
  ],
  [
   960,
   0.1644
  ],
  [
   960,
   0.2951
  ],
  [
   960,
   0.2479
  ],
  [
   960,
   0.2099
  ],
  [
   960,
   0.2323
  ],
  [
   960,
   0.1504
  ],
  [
   960,
   0.3293
  ],
  [
   960,
   0.3633
  ],
  [
   960,
   0.3073
  ],
  [
   960,
   0.3164
  ],
  [
   960,
   0.3301
  ],
  [
   960,
   0.3974
  ],
  [
   960,
   0.3726
  ],
  [
   960,
   0.1616
  ],
  [
   960,
   0.157
  ],
  [
   960,
   0.2991
  ],
  [
   960,
   0.2084
  ],
  [
   960,
   0.1889
  ],
  [
   960,
   0.2487
  ],
  [
   960,
   0.2485
  ],
  [
   960,
   0.1458
  ],
  [
   960,
   0.2454
  ],
  [
   960,
   0.3324
  ],
  [
   960,
   0.2478
  ],
  [
   960,
   0.2262
  ],
  [
   960,
   0.2575
  ],
  [
   960,
   0.2405
  ],
  [
   960,
   0.1937
  ],
  [
   960,
   0.2271
  ],
  [
   960,
   0.1138
  ],
  [
   960,
   0.3416
  ],
  [
   960,
   0.1619
  ],
  [
   960,
   0.242
  ],
  [
   960,
   0.177
  ],
  [
   960,
   0.1778
  ],
  [
   960,
   0.1901
  ],
  [
   960,
   0.2288
  ],
  [
   960,
   0.1428
  ],
  [
   960,
   0.1532
  ],
  [
   960,
   0.3108
  ],
  [
   960,
   0.218
  ],
  [
   960,
   0.1687
  ],
  [
   960,
   0.2205
  ],
  [
   960,
   0.3165
  ],
  [
   960,
   0.2606
  ],
  [
   960,
   0.2219
  ],
  [
   960,
   0.2137
  ],
  [
   960,
   0.2336
  ],
  [
   960,
   0.2239
  ],
  [
   960,
   0.2104
  ],
  [
   960,
   0.1352
  ],
  [
   960,
   0.16
  ],
  [
   960,
   0.2476
  ],
  [
   960,
   0.1884
  ],
  [
   960,
   0.1825
  ],
  [
   960,
   0.119
  ],
  [
   960,
   0.1092
  ],
  [
   960,
   0.1187
  ],
  [
   960,
   0.1874
  ],
  [
   960,
   0.1939
  ],
  [
   960,
   0.1704
  ],
  [
   960,
   0.1549
  ],
  [
   960,
   0.223
  ],
  [
   960,
   0.2901
  ],
  [
   960,
   0.2404
  ],
  [
   960,
   0.2185
  ],
  [
   960,
   0.2175
  ],
  [
   960,
   0.2147
  ],
  [
   960,
   0.1741
  ],
  [
   960,
   0.1591
  ],
  [
   960,
   0.1416
  ],
  [
   960,
   0.1739
  ],
  [
   960,
   0.1343
  ],
  [
   960,
   0.1259
  ],
  [
   960,
   0.1349
  ],
  [
   960,
   0.1214
  ],
  [
   960,
   0.0943
  ],
  [
   960,
   0.1145
  ],
  [
   960,
   0.3483
  ],
  [
   960,
   0.2424
  ],
  [
   960,
   0.2214
  ],
  [
   960,
   0.1945
  ],
  [
   960,
   0.2687
  ],
  [
   960,
   0.27
  ],
  [
   960,
   0.1635
  ],
  [
   960,
   0.1737
  ],
  [
   960,
   0.2004
  ],
  [
   960,
   0.175
  ],
  [
   960,
   0.142
  ],
  [
   960,
   0.1522
  ],
  [
   960,
   0.2417
  ],
  [
   960,
   0.2184
  ],
  [
   960,
   0.123
  ],
  [
   960,
   0.1679
  ],
  [
   960,
   0.1451
  ],
  [
   960,
   0.079
  ],
  [
   960,
   0.3009
  ],
  [
   960,
   0.1634
  ],
  [
   960,
   0.1956
  ],
  [
   960,
   0.1285
  ],
  [
   960,
   0.1406
  ],
  [
   960,
   0.1398
  ],
  [
   960,
   0.0885
  ],
  [
   960,
   0.1096
  ],
  [
   960,
   0.0952
  ],
  [
   960,
   0.26
  ],
  [
   960,
   0.2966
  ],
  [
   960,
   0.2747
  ],
  [
   960,
   0.2741
  ],
  [
   960,
   0.1136
  ],
  [
   960,
   0.1086
  ],
  [
   960,
   0.1997
  ],
  [
   960,
   0.1983
  ],
  [
   960,
   0.1679
  ],
  [
   960,
   0.2174
  ],
  [
   960,
   0.1715
  ],
  [
   960,
   0.2232
  ],
  [
   960,
   0.2139
  ],
  [
   960,
   0.2013
  ],
  [
   960,
   0.3094
  ],
  [
   960,
   0.1249
  ],
  [
   960,
   0.1172
  ],
  [
   960,
   0.177
  ],
  [
   960,
   0.1754
  ],
  [
   960,
   0.2171
  ],
  [
   960,
   0.2055
  ],
  [
   960,
   0.1256
  ],
  [
   960,
   0.1384
  ],
  [
   960,
   0.1967
  ],
  [
   960,
   0.1903
  ],
  [
   960,
   0.1512
  ],
  [
   960,
   0.1594
  ],
  [
   960,
   0.1559
  ],
  [
   960,
   0.148
  ],
  [
   960,
   0.1128
  ],
  [
   960,
   0.1847
  ],
  [
   960,
   0.1434
  ],
  [
   960,
   0.1157
  ],
  [
   960,
   0.1895
  ],
  [
   960,
   0.1451
  ],
  [
   960,
   0.2059
  ],
  [
   960,
   0.2426
  ],
  [
   960,
   0.1927
  ],
  [
   960,
   0.1461
  ],
  [
   960,
   0.1472
  ],
  [
   960,
   0.1357
  ],
  [
   960,
   0.1416
  ],
  [
   960,
   0.2199
  ],
  [
   960,
   0.2298
  ],
  [
   960,
   0.2473
  ],
  [
   960,
   0.1293
  ],
  [
   960,
   0.1929
  ],
  [
   960,
   0.1462
  ],
  [
   960,
   0.1391
  ],
  [
   960,
   0.2051
  ],
  [
   960,
   0.1839
  ],
  [
   960,
   0.2176
  ],
  [
   960,
   0.2386
  ],
  [
   960,
   0.124
  ],
  [
   960,
   0.3373
  ],
  [
   960,
   0.1971
  ],
  [
   960,
   0.2051
  ],
  [
   960,
   0.1925
  ],
  [
   960,
   0.2195
  ],
  [
   960,
   0.3385
  ],
  [
   960,
   0.22
  ],
  [
   960,
   0.2296
  ],
  [
   960,
   0.4608
  ],
  [
   960,
   0.4191
  ],
  [
   960,
   0.5259
  ],
  [
   960,
   0.3538
  ],
  [
   960,
   0.4894
  ],
  [
   960,
   0.5009
  ],
  [
   960,
   0.2108
  ],
  [
   960,
   0.5432
  ],
  [
   960,
   0.5185
  ],
  [
   960,
   0.4072
  ],
  [
   960,
   0.1422
  ],
  [
   960,
   0.2478
  ],
  [
   960,
   0.2984
  ],
  [
   960,
   0.334
  ],
  [
   960,
   0.4128
  ],
  [
   960,
   0.3777
  ],
  [
   960,
   0.3231
  ],
  [
   960,
   0.4508
  ],
  [
   960,
   0.2409
  ],
  [
   960,
   0.4313
  ],
  [
   960,
   0.4242
  ],
  [
   960,
   0.3648
  ],
  [
   960,
   0.5099
  ],
  [
   960,
   0.2885
  ],
  [
   960,
   0.3457
  ],
  [
   960,
   0.3749
  ],
  [
   960,
   0.4106
  ],
  [
   960,
   0.3813
  ],
  [
   960,
   0.4998
  ],
  [
   960,
   0.411
  ],
  [
   960,
   0.3287
  ],
  [
   960,
   0.1837
  ],
  [
   960,
   0.2407
  ],
  [
   960,
   0.3845
  ],
  [
   960,
   0.2058
  ],
  [
   850,
   0.2586
  ],
  [
   960,
   0.3941
  ],
  [
   960,
   0.2423
  ],
  [
   960,
   0.2674
  ],
  [
   960,
   0.2838
  ],
  [
   960,
   0.3186
  ],
  [
   960,
   0.2437
  ],
  [
   960,
   0.4251
  ],
  [
   960,
   0.281
  ],
  [
   960,
   0.282
  ],
  [
   960,
   0.2279
  ],
  [
   960,
   0.2275
  ],
  [
   960,
   0.3632
  ],
  [
   960,
   0.243
  ],
  [
   960,
   0.2647
  ],
  [
   960,
   0.2869
  ],
  [
   960,
   0.2935
  ],
  [
   960,
   0.2882
  ],
  [
   960,
   0.3144
  ],
  [
   960,
   0.3143
  ],
  [
   960,
   0.308
  ],
  [
   960,
   0.4773
  ],
  [
   960,
   0.4513
  ],
  [
   960,
   0.2924
  ],
  [
   960,
   0.25
  ],
  [
   775,
   0.3865
  ],
  [
   960,
   0.3512
  ],
  [
   960,
   0.4247
  ],
  [
   960,
   0.2292
  ],
  [
   960,
   0.3752
  ],
  [
   960,
   0.1929
  ],
  [
   960,
   0.262
  ],
  [
   960,
   0.3589
  ],
  [
   960,
   0.1799
  ],
  [
   960,
   0.3673
  ],
  [
   960,
   0.3567
  ],
  [
   960,
   0.3504
  ],
  [
   960,
   0.21
  ],
  [
   960,
   0.2271
  ],
  [
   960,
   0.2064
  ],
  [
   960,
   0.3507
  ],
  [
   960,
   0.2758
  ],
  [
   960,
   0.3297
  ],
  [
   960,
   0.3029
  ],
  [
   960,
   0.3971
  ],
  [
   960,
   0.3547
  ],
  [
   960,
   0.3624
  ],
  [
   960,
   0.2546
  ],
  [
   960,
   0.3027
  ],
  [
   960,
   0.2873
  ],
  [
   960,
   0.3792
  ],
  [
   960,
   0.368
  ],
  [
   960,
   0.3491
  ],
  [
   960,
   0.2466
  ],
  [
   960,
   0.3694
  ],
  [
   960,
   0.2904
  ],
  [
   960,
   0.4
  ],
  [
   960,
   0.3913
  ],
  [
   960,
   0.3589
  ],
  [
   960,
   0.357
  ],
  [
   960,
   0.2396
  ],
  [
   960,
   0.3748
  ],
  [
   960,
   0.3286
  ],
  [
   960,
   0.3108
  ],
  [
   960,
   0.3614
  ],
  [
   960,
   0.3962
  ],
  [
   960,
   0.2303
  ],
  [
   960,
   0.3171
  ],
  [
   960,
   0.2489
  ],
  [
   960,
   0.3079
  ],
  [
   960,
   0.2704
  ],
  [
   960,
   0.2669
  ],
  [
   960,
   0.3163
  ],
  [
   960,
   0.266
  ],
  [
   960,
   0.2774
  ],
  [
   960,
   0.2395
  ],
  [
   960,
   0.2387
  ],
  [
   960,
   0.3041
  ],
  [
   960,
   0.1895
  ],
  [
   960,
   0.247
  ],
  [
   960,
   0.2083
  ],
  [
   960,
   0.2559
  ],
  [
   960,
   0.2788
  ],
  [
   960,
   0.2945
  ],
  [
   960,
   0.271
  ],
  [
   960,
   0.279
  ],
  [
   960,
   0.3051
  ],
  [
   960,
   0.3084
  ],
  [
   960,
   0.312
  ],
  [
   960,
   0.2904
  ],
  [
   960,
   0.1925
  ],
  [
   960,
   0.2951
  ],
  [
   960,
   0.1936
  ],
  [
   960,
   0.2875
  ],
  [
   960,
   0.2611
  ],
  [
   960,
   0.3235
  ],
  [
   960,
   0.2208
  ],
  [
   960,
   0.2446
  ],
  [
   960,
   0.1606
  ],
  [
   960,
   0.1687
  ],
  [
   960,
   0.3063
  ],
  [
   350,
   0.2868
  ],
  [
   640,
   0.1814
  ],
  [
   960,
   0.1916
  ],
  [
   960,
   0.2518
  ],
  [
   789,
   0.2361
  ],
  [
   960,
   0.2145
  ],
  [
   960,
   0.2072
  ],
  [
   960,
   0.2882
  ],
  [
   960,
   0.1627
  ],
  [
   787,
   0.5323
  ],
  [
   612,
   0.3636
  ],
  [
   612,
   0.3677
  ]
 ]
}

TURLER = [t for t in KOMUTLAR if not t.startswith("_")]
print("komut bankası:", {t: len(KOMUTLAR[t]) for t in TURLER})
print("biçim profili:", BICIM_PROFILI["_olcum"])

## 4 · Komut birleştirme ve biçim oturtma

37 temel sahne, fotoğrafçılık değiştiricileriyle çarpılarak binlerce ayrı
komut üretir. Değiştiriciler **bilinçli olarak fotografiktir**: `cinematic`,
`8k`, `artstation` gibi ifadeler modeli konsept sanatına iter ve ölçmek
istediğimiz alanı kaçırır.

In [ ]:
import hashlib
import io
import os
import random

from PIL import Image

# Fotoğrafçılık değiştiricileri — hepsi kadraj/ışık/doku, hiçbiri stil değil.
CERCEVE = [
    "wide shot", "medium shot", "tight shot", "eye-level",
    "slightly low angle", "shot from across the street",
    "shot from a doorway", "elevated view from an upper window",
]
KOSUL = [
    "overcast", "bright midday sun", "late afternoon light", "grey drizzle",
    "dust haze in the air", "early morning light", "blue hour", "harsh backlight",
]
DOKU = [
    "slight motion blur", "visible sensor noise", "shallow depth of field",
    "flat muted colours", "compressed shadows", "slightly underexposed", "",
]

# SD/SDXL için olumsuz komut. İki iş yapar: (1) modeli fotoğraf tarafında
# tutar, (2) pilot koşuda görülen "kadraja telefon tutan el" kusurunu bastırır.
OLUMSUZ = (
    "illustration, painting, drawing, anime, cartoon, 3d render, cgi, "
    "concept art, digital art, watermark, text, caption, logo, signature, "
    "oversaturated, hdr, hand in frame, holding a phone, selfie, "
    "smartphone screen, collage, border, frame"
)


def komut_uret(tur: str, sira: int, rastgele: random.Random) -> str:
    """Temel sahneyi değiştiricilerle birleştirir."""
    taban = KOMUTLAR[tur][sira % len(KOMUTLAR[tur])]
    parcalar = [taban, rastgele.choice(CERCEVE), rastgele.choice(KOSUL)]
    if (doku := rastgele.choice(DOKU)):
        parcalar.append(doku)
    return ", ".join(parcalar)


def tur_plani(toplam: int, rastgele: random.Random) -> list:
    """Tür dağılımını ağırlıklara göre kurar ve karıştırır."""
    plan = []
    for tur, agirlik in TUR_AGIRLIK.items():
        plan += [tur] * round(toplam * agirlik)
    while len(plan) < toplam:
        plan.append(TURLER[0])
    plan = plan[:toplam]
    rastgele.shuffle(plan)
    return plan


_CIFTLER = BICIM_PROFILI["ciftler"]


def _hedef_bicim(ad: str):
    """Dosya adından türetilmiş, yeniden üretilebilir (genişlik, bayt/piksel)."""
    h = int(hashlib.blake2b(ad.encode("utf-8"), digest_size=8).hexdigest(), 16)
    return _CIFTLER[h % len(_CIFTLER)]


def bicime_oturt(im: Image.Image, yol: str, ad: str) -> dict:
    """Görseli gerçek korpusun biçim hedefine oturtup TEK KEZ JPEG kodlar."""
    im = im.convert("RGB")
    ham = im.size
    hedef_en, hedef_bpp = _hedef_bicim(ad)
    if im.width != hedef_en:
        oran = hedef_en / im.width
        im = im.resize((hedef_en, max(1, round(im.height * oran))), Image.LANCZOS)

    piksel = im.width * im.height
    alt, ust, en_iyi, en_iyi_fark = 30, 96, 96, float("inf")
    while alt <= ust:
        orta = (alt + ust) // 2
        tampon = io.BytesIO()
        im.save(tampon, "JPEG", quality=orta, optimize=True)
        bpp = tampon.tell() / piksel
        if (fark := abs(bpp - hedef_bpp)) < en_iyi_fark:
            en_iyi, en_iyi_fark = orta, fark
        if bpp < hedef_bpp:
            alt = orta + 1
        else:
            ust = orta - 1

    im.save(yol, "JPEG", quality=en_iyi, optimize=True)
    bayt = os.path.getsize(yol)
    return {
        "boyut": f"{im.width}x{im.height}",
        "uretim_boyutu": f"{ham[0]}x{ham[1]}",
        "jpeg_kalite": en_iyi,
        "hedef_bayt_piksel": hedef_bpp,
        "bayt": bayt,
        "bayt_piksel": round(bayt / piksel, 4),
    }


print("komut örneği:")
_r = random.Random(TOHUM)
for _t in TURLER:
    print(f"  [{_t}] {komut_uret(_t, 0, _r)[:150]}…")

## 5 · Üretim motoru

Her üretici kendi hücresinde çalışır. Bir hücre çökerse (OOM, indirme hatası)
diğerleri etkilenmez; kayıt dosyası her üreticiden sonra güncellenir, yani
yarıda kesilse bile o ana kadar üretilenler korunur.

In [ ]:
import gc
import json
import os
import random
import time
import traceback

KAYIT_YOLU = f"{CIKTI_DIZINI}/uretim_kayit.json"


def kayitlari_oku() -> list:
    if os.path.exists(KAYIT_YOLU):
        with open(KAYIT_YOLU, encoding="utf-8") as f:
            return json.load(f)
    return []


def kayitlari_yaz(kayitlar: list) -> None:
    with open(KAYIT_YOLU, "w", encoding="utf-8") as f:
        json.dump(kayitlar, f, ensure_ascii=False, indent=1)


def uret(kod_adi: str, boru, uretici_adi: str, lisans: str, rol: str,
         adet: int, cagir, mimari: str) -> None:
    """Bir üreticiyle `adet` görsel üretir ve kayda işler.

    `cagir(komut, tohum)` tek bir PIL görüntüsü döndürmelidir.
    """
    kayitlar = kayitlari_oku()
    varolan = {k["dosya"] for k in kayitlar}
    rastgele = random.Random(f"{TOHUM}-{kod_adi}")
    plan = tur_plani(adet, rastgele)

    basladi = time.time()
    uretilen = 0
    for i, tur in enumerate(plan):
        ad = f"SYNTH_{tur.replace('ı','i')}_{kod_adi}_{i:04d}.jpg"
        if ad in varolan:
            continue
        komut = komut_uret(tur, i, rastgele)
        tohum = TOHUM + i
        try:
            im = cagir(komut, tohum)
        except Exception:
            print(f"  🔴 {ad} üretilemedi:")
            traceback.print_exc(limit=1)
            continue

        yol = f"{CIKTI_DIZINI}/goruntuler/{ad}"
        olcum = bicime_oturt(im, yol, ad)
        kayitlar.append({
            "dosya": ad, "tur": tur, "uretici": uretici_adi, "kod": kod_adi,
            "mimari": mimari, "lisans": lisans, "rol": rol,
            "komut": komut, "tohum": tohum, "cerceve_kusuru": False, **olcum,
        })
        uretilen += 1

        if uretilen % 25 == 0:
            kayitlari_yaz(kayitlar)
            gecen = time.time() - basladi
            hiz = gecen / uretilen
            kalan = (len(plan) - i - 1) * hiz
            print(f"  {uretilen}/{adet}  ·  {hiz:.1f} sn/görsel  ·  "
                  f"kalan ~{kalan/60:.0f} dk")

    kayitlari_yaz(kayitlar)
    print(f"✓ {uretici_adi}: {uretilen} görsel · "
          f"{(time.time()-basladi)/60:.1f} dk · rol={rol}")


def bosalt(*nesneler):
    """Boru hattını hafızadan düşürür — sıradaki model sığsın diye."""
    for n in nesneler:
        del n
    gc.collect()
    torch.cuda.empty_cache()
    print(f"VRAM boşta: {torch.cuda.mem_get_info()[0]/1e9:.1f} GB")

## 6 · SANA 1.6B  ·  **eğitim**  ·  Apache 2.0

Linear-attention DiT — SDXL'in UNet'inden bütünüyle farklı bir mimari,
dolayısıyla farklı bir artefakt imzası. Lisansı **Apache 2.0**: eğitimde
kullanılması ağırlığın köken zincirini temiz bırakır.

`complex_human_instruction=None` veriyoruz: SANA varsayılan olarak istemi bir
dil modeliyle "zenginleştiriyor" ve kısa fotoğrafçılık istemlerimizi edebî
sahne betimlerine çeviriyor — tam da kaçındığımız şey.

In [ ]:
from diffusers import SanaPipeline

boru = SanaPipeline.from_pretrained(
    "Efficient-Large-Model/Sana_1600M_1024px_diffusers",
    variant="fp16", torch_dtype=torch.float16)
boru.to("cuda")
boru.vae.to(torch.bfloat16)
boru.text_encoder.to(torch.bfloat16)
boru.set_progress_bar_config(disable=True)

def _sana(komut, tohum):
    return boru(
        prompt=komut, negative_prompt=OLUMSUZ,
        num_inference_steps=20, guidance_scale=4.5,
        height=768, width=1024,
        complex_human_instruction=None,   # istem zenginleştirmeyi kapat
        generator=torch.Generator("cuda").manual_seed(tohum),
    ).images[0]

uret("sana-1600m", boru, "Efficient-Large-Model/Sana_1600M_1024px_diffusers",
     "Apache-2.0", "egitim", ADET["sana-1600m"], _sana, "linear-attention DiT")
bosalt(boru)

## 7 · SDXL base 1.0  ·  **eğitim**  ·  OpenRAIL++-M

`SADECE_TEMIZ_LISANS = True` ise bu hücre kendini atlar.

In [ ]:
if SADECE_TEMIZ_LISANS:
    print("⏭  atlandı: SADECE_TEMIZ_LISANS açık, SDXL OpenRAIL++-M lisanslı")
else:
    from diffusers import StableDiffusionXLPipeline

    boru = StableDiffusionXLPipeline.from_pretrained(
        "stabilityai/stable-diffusion-xl-base-1.0",
        torch_dtype=torch.float16, variant="fp16", use_safetensors=True).to("cuda")
    boru.set_progress_bar_config(disable=True)

    def _sdxl(komut, tohum):
        return boru(
            prompt=komut, negative_prompt=OLUMSUZ,
            num_inference_steps=30, guidance_scale=6.0,
            height=768, width=1024,
            generator=torch.Generator("cuda").manual_seed(tohum),
        ).images[0]

    uret("sdxl-base", boru, "stabilityai/stable-diffusion-xl-base-1.0",
         "CreativeML OpenRAIL++-M", "egitim", ADET["sdxl-base"], _sdxl,
         "UNet latent diffusion")
    bosalt(boru)

## 8 · PixArt-Σ XL-2  ·  **TUTULAN** (ölçüm)  ·  OpenRAIL++-M

⚠️ **Bu üreticinin görselleri eğitime GİRMEZ.** Ölçümde "görülmemiş üretici"
rolünü oynarlar. Kayıtta `rol="tutulan"` ile işaretlidirler ve eğitim betiği
o kayıtları dışarıda bırakır.

Forged Calamity (2026) makalesinin bulgusu bu seçimi destekliyor: ince ayarlı
dedektörler PixArt'la üretilmiş görsellerde %0–12'ye düşüyor. Yani PixArt,
tutulan üretici olarak **en zorlayıcı** seçenek — ölçümü kolaylaştırmıyor,
zorlaştırıyor. Doğru olan da bu.

In [ ]:
from diffusers import PixArtSigmaPipeline

boru = PixArtSigmaPipeline.from_pretrained(
    "PixArt-alpha/PixArt-Sigma-XL-2-1024-MS", torch_dtype=torch.float16)
boru.enable_model_cpu_offload()   # T5 metin kodlayıcısı ~19 GB
boru.set_progress_bar_config(disable=True)

def _pixart(komut, tohum):
    return boru(
        prompt=komut, negative_prompt=OLUMSUZ,
        num_inference_steps=20, guidance_scale=4.5,
        height=768, width=1024,
        generator=torch.Generator("cuda").manual_seed(tohum),
    ).images[0]

uret("pixart-sigma", boru, "PixArt-alpha/PixArt-Sigma-XL-2-1024-MS",
     "CreativeML OpenRAIL++-M", "tutulan", ADET["pixart-sigma"], _pixart,
     "DiT + T5")
bosalt(boru)

## 8b · Hizalı sahteler  ·  **eğitim**  ·  kendi korpusumuzdan

Kör noktayı asıl kapatan parça. Sahte, **gerçek afet fotoğrafının kendisinden**
üretilir; içerik birebir aynı kalır, geriye kalan tek fark üretim izidir.

**Önce afet korpusunu yükleyin.** Yerelde:

```bash
cd "KrizKalkanAI/data/external/provenance"
zip -r ~/Desktop/afet_korpusu.zip goruntuler kayitlar.jsonl
```

Sonra aşağıdaki hücre dosya seçme kutusu açar.

⚠️ **Olay bazlı ayrım burada uygulanır.** Hizalı sahteler yalnızca *eğitim*
olaylarından üretilir. Tutulan olaylara (aşağıda listeli) dokunulmaz: o
fotoğraflar kabul kapısının ölçüm kümesidir ve içerikleri eğitime hiçbir
biçimde sızmamalıdır.

In [ ]:
from google.colab import files

print("afet_korpusu.zip dosyasını seçin:")
yuklenen = files.upload()
ad = next(iter(yuklenen))
!unzip -q -o "{ad}" -d /content/afet_korpusu
!ls /content/afet_korpusu

In [ ]:
OLAY_AYRIMI = {
 "_aciklama": "Afet korpusunun OLAY BAZLI ayrımı. Tutulan olaylar eğitime hiç girmez ve kabul kapısı (afet_ozgulluk) YALNIZCA orada ölçülür.",
 "_gerekce": "Önceki eğitim koşusu korpusun %80'ini rastgele eğitime koyuyordu, ama afet_ozgulluk korpusun TAMAMINDA ölçülüyordu — yani kapının baktığı fotoğrafların çoğunu model eğitimde görmüştü ve 0,9885 iyimser bir sayıydı. Rastgele bölme de yetmez: aynı olayın fotoğrafları birbirine çok benzer, rastgele bölmede aynı enkazın başka karesi hem eğitimde hem ölçümde çıkar.",
 "_olcum": {
  "toplam": 786,
  "tutulan": 130,
  "tutulan_oran": 0.1654
 },
 "tutulan_olaylar": [
  "Türkiye'de depremler (arşiv)",
  "2023 Adıyaman-Şanlıurfa sel felaketi",
  "2021 Milas yangını",
  "2022 Ankara selleri"
 ],
 "olay_sayilari": {
  "6 Şubat 2023 Kahramanmaraş depremleri": 610,
  "20 Şubat 2023 Hatay depremi": 3,
  "2021 Milas yangını": 14,
  "2023 Adıyaman-Şanlıurfa sel felaketi": 17,
  "2022 Ankara selleri": 1,
  "Türkiye'de seller (arşiv)": 43,
  "Türkiye'de depremler (arşiv)": 98
 }
}

TUTULAN_OLAYLAR = set(OLAY_AYRIMI["tutulan_olaylar"])
print("tutulan olaylar (eğitime GİRMEZ):")
for o in sorted(TUTULAN_OLAYLAR):
    print(f"  {OLAY_AYRIMI['olay_sayilari'].get(o, 0):4d}  {o}")
print(f"\ntoplam tutulan: {OLAY_AYRIMI['_olcum']['tutulan']}"
      f"/{OLAY_AYRIMI['_olcum']['toplam']}"
      f" (%{OLAY_AYRIMI['_olcum']['tutulan_oran']*100:.1f})")

In [ ]:
import glob
import json
import os
import random
import time

import torch
from PIL import Image

if HIZALI_ADET <= 0:
    print("⏭  atlandı: HIZALI_ADET = 0")
else:
    import numpy as np
    from diffusers import AutoPipelineForImage2Image
    from diffusers.models import AutoencoderKL

    # Korpus kaydını bul (zip iç düzeni değişebilir)
    aday = glob.glob("/content/afet_korpusu/**/kayitlar.jsonl", recursive=True)
    assert aday, "kayitlar.jsonl bulunamadı — zip'i doğru kurduğunuzdan emin olun"
    kayit_yolu = aday[0]
    korpus_kok = os.path.dirname(kayit_yolu)

    with open(kayit_yolu, encoding="utf-8") as f:
        korpus = [json.loads(s) for s in f if s.strip()]

    # OLAY BAZLI AYRIM — tutulan olaylar dışarıda kalır
    egitim_foto = [
        k for k in korpus
        if k["olay"] not in TUTULAN_OLAYLAR
        and os.path.exists(os.path.join(korpus_kok, "goruntuler", k["dosya"]))
    ]
    atilan = len(korpus) - len(egitim_foto)
    print(f"korpus {len(korpus)} · eğitime uygun {len(egitim_foto)} · "
          f"tutulan/eksik {atilan}")
    assert egitim_foto, "eğitim olaylarından hiç fotoğraf bulunamadı"

    rastgele = random.Random(f"{TOHUM}-hizali")
    rastgele.shuffle(egitim_foto)
    secim = egitim_foto[:HIZALI_ADET]
    yari = len(secim) // 2

    vae = AutoencoderKL.from_pretrained(
        "stabilityai/sdxl-vae", torch_dtype=torch.float32).to("cuda").eval()

    def _vae_yeniden(yol):
        """Fotoğrafı SDXL sıkıştırıcısından geçirip geri açar."""
        im = Image.open(yol).convert("RGB")
        im.thumbnail((1024, 1024))
        w, h = (im.width // 8) * 8, (im.height // 8) * 8
        im = im.crop((0, 0, w, h))
        x = torch.from_numpy(np.asarray(im, dtype=np.float32) / 127.5 - 1.0)
        x = x.permute(2, 0, 1)[None].to("cuda")
        with torch.no_grad():
            geri = vae.decode(vae.encode(x).latent_dist.mode()).sample
        dizi = ((geri[0].permute(1, 2, 0).clamp(-1, 1).float().cpu().numpy() + 1) * 127.5)
        return Image.fromarray(dizi.astype(np.uint8))

    kayitlar = kayitlari_oku()
    varolan = {k["dosya"] for k in kayitlar}
    uretilen = 0

    # ── (a) VAE yeniden kurma ──
    for i, foto in enumerate(secim[:yari]):
        ad = f"SYNTH_hizali-vae_{i:04d}.jpg"
        if ad in varolan:
            continue
        try:
            im = _vae_yeniden(os.path.join(korpus_kok, "goruntuler", foto["dosya"]))
        except Exception as e:
            print(f"  🔴 {foto['dosya']}: {e}")
            continue
        yol = f"{CIKTI_DIZINI}/goruntuler/{ad}"
        olcum = bicime_oturt(im, yol, ad)
        kayitlar.append({
            "dosya": ad, "tur": "hizali", "uretici": "stabilityai/sdxl-vae",
            "kod": "hizali-vae", "mimari": "VAE yeniden kurma",
            "lisans": "CreativeML OpenRAIL++-M (VAE) + korpus lisansı",
            "rol": "egitim", "komut": "", "tohum": TOHUM + i,
            "kaynak_foto": foto["dosya"], "kaynak_olay": foto["olay"],
            "cerceve_kusuru": False, **olcum,
        })
        uretilen += 1
        if uretilen % 50 == 0:
            kayitlari_yaz(kayitlar)
            print(f"  vae {uretilen}/{yari}")
    kayitlari_yaz(kayitlar)
    print(f"✓ VAE yeniden kurma: {uretilen} görsel")
    bosalt(vae)

    # ── (b) img2img ──
    boru = AutoPipelineForImage2Image.from_pretrained(
        "stabilityai/stable-diffusion-xl-base-1.0",
        torch_dtype=torch.float16, variant="fp16", use_safetensors=True).to("cuda")
    boru.set_progress_bar_config(disable=True)

    basladi = time.time()
    img_uretilen = 0
    for i, foto in enumerate(secim[yari:]):
        ad = f"SYNTH_hizali-img2img_{i:04d}.jpg"
        if ad in varolan:
            continue
        kaynak = Image.open(
            os.path.join(korpus_kok, "goruntuler", foto["dosya"])).convert("RGB")
        kaynak.thumbnail((1024, 1024))
        w, h = (kaynak.width // 8) * 8, (kaynak.height // 8) * 8
        kaynak = kaynak.crop((0, 0, w, h))
        # İstem sahneyi TARİF ETMEZ, yalnızca fotoğrafik kalmasını sağlar:
        # amaç içeriği korumak, yeniden kurgulamak değil.
        try:
            im = boru(
                prompt="a photograph", negative_prompt=OLUMSUZ,
                image=kaynak, strength=0.35, guidance_scale=5.0,
                num_inference_steps=30,
                generator=torch.Generator("cuda").manual_seed(TOHUM + i),
            ).images[0]
        except Exception as e:
            print(f"  🔴 {foto['dosya']}: {e}")
            continue
        yol = f"{CIKTI_DIZINI}/goruntuler/{ad}"
        olcum = bicime_oturt(im, yol, ad)
        kayitlar.append({
            "dosya": ad, "tur": "hizali",
            "uretici": "stabilityai/stable-diffusion-xl-base-1.0 (img2img)",
            "kod": "hizali-img2img", "mimari": "UNet latent diffusion · img2img 0.35",
            "lisans": "CreativeML OpenRAIL++-M + korpus lisansı",
            "rol": "egitim", "komut": "a photograph", "tohum": TOHUM + i,
            "kaynak_foto": foto["dosya"], "kaynak_olay": foto["olay"],
            "cerceve_kusuru": False, **olcum,
        })
        img_uretilen += 1
        if img_uretilen % 25 == 0:
            kayitlari_yaz(kayitlar)
            print(f"  img2img {img_uretilen}/{len(secim)-yari} · "
                  f"{(time.time()-basladi)/img_uretilen:.1f} sn/görsel")
    kayitlari_yaz(kayitlar)
    print(f"✓ img2img: {img_uretilen} görsel")
    bosalt(boru)

    # Tutulan olayların hiçbir hizalı sahteye kaynaklık etmediğini doğrula.
    sizinti = [k for k in kayitlari_oku()
               if k.get("kaynak_olay") in TUTULAN_OLAYLAR]
    assert not sizinti, f"🔴 TUTULAN OLAY SIZINTISI: {len(sizinti)} kayıt"
    print("✓ tutulan olaylardan sızıntı yok")

## 9 · Özet ve denetim

In [ ]:
import collections, statistics

kayitlar = kayitlari_oku()
print(f"toplam {len(kayitlar)} görsel\n")

for alan in ("uretici", "tur", "rol", "lisans"):
    print(alan.upper())
    for k, v in collections.Counter(x[alan] for x in kayitlar).most_common():
        print(f"  {v:5d}  {k}")
    print()

bpp = [k["bayt_piksel"] for k in kayitlar]
kal = [k["jpeg_kalite"] for k in kayitlar]
print(f"bayt/piksel  medyan {statistics.median(bpp):.3f}   "
      f"(gerçek korpus {BICIM_PROFILI['_olcum']['bayt_piksel_medyan']})")
print(f"JPEG kalite  medyan {statistics.median(kal):.0f}  "
      f"min {min(kal)}  max {max(kal)}")

# Rol ayrımının bozulmadığını doğrula: aynı üretici iki rolde olamaz.
roller = collections.defaultdict(set)
for k in kayitlar:
    roller[k["uretici"]].add(k["rol"])
bozuk = {u: r for u, r in roller.items() if len(r) > 1}
assert not bozuk, f"🔴 rol ayrımı bozuk: {bozuk}"
print("\n✓ üretici bazlı rol ayrımı sağlam")

## 10 · Gözle kontrol

Fotogerçekçilik ve kadraj kusuru için örnek bir tabaka. **Bakılacak şey:**
kadraja giren el/telefon, metin/filigran, çizim üslubu. Kusurlu kareler
yerelde `cerceve_kusuru` ile işaretlenip ölçümde ayrı raporlanır.

In [ ]:
from PIL import Image
import random as _rnd

kayitlar = kayitlari_oku()
_rnd.seed(0)
ornek = _rnd.sample(kayitlar, min(24, len(kayitlar)))
K, S = 260, 6
satir = (len(ornek) + S - 1) // S
tabaka = Image.new("RGB", (K*S, satir*K), (18, 18, 18))
for j, kayit in enumerate(ornek):
    im = Image.open(f"{CIKTI_DIZINI}/goruntuler/{kayit['dosya']}").convert("RGB")
    im.thumbnail((K, K))
    tabaka.paste(im, ((j % S)*K + (K-im.width)//2, (j // S)*K + (K-im.height)//2))
tabaka

## 11 · Paketle ve indir

Zip'i bilgisayarınıza indirin. Ömer'e/depoya şu komutla girer:

```bash
unzip sentetik_afet.zip -d /tmp/sentetik_ham
python scripts/data/build_sentetik_korpus.py \
    --ham /tmp/sentetik_ham/goruntuler \
    --kayit /tmp/sentetik_ham/uretim_kayit.json \
    --bicim-uygulanmis
```

`--bicim-uygulanmis` önemlidir: biçim bu defterde zaten uygulandı, yerelde
ikinci kez kodlamak her dosyaya çift sıkıştırma izi bindirirdi.

In [ ]:
import shutil
from google.colab import files

arsiv = shutil.make_archive("/content/sentetik_afet", "zip", CIKTI_DIZINI)
print(f"paket: {arsiv} · {os.path.getsize(arsiv)/1e6:.0f} MB")
files.download(arsiv)